# Banking Loan Applications: Data Cleaning and EDA

This notebook checks the original loan application data, fixes common data quality issues, and saves a clean copy. I have kept the steps simple and used pandas methods covered in the course.

## 1. Import pandas and load the data

In [ ]:
import pandas as pd

df = pd.read_csv('../data/private/banking_loan_applications_messy.csv')
df.head()

## 2. Understand the data

I am checking the number of rows, column types, missing values, and the summary of numeric columns before making changes.

In [ ]:
df.shape
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df['application_id'].duplicated().sum()

## 3. Clean the data

First I remove duplicate applications and tidy text columns. I replace values outside reasonable ranges with missing values so they can be filled consistently below.

In [ ]:
# Remove duplicate application records
df = df.drop_duplicates(subset='application_id', keep='first').copy()

# Remove extra spaces and make text values consistent
text_columns = df.select_dtypes(include='str').columns
for column in text_columns:
    df[column] = df[column].str.strip()

df['property_area'] = df['property_area'].str.title()
df['loan_status'] = df['loan_status'].str.title()

In [ ]:
# Replace invalid numeric values with missing values
df.loc[(df['applicant_age'] < 18) | (df['applicant_age'] > 100), 'applicant_age'] = pd.NA
df.loc[df['monthly_income_inr'] < 0, 'monthly_income_inr'] = pd.NA
df.loc[(df['credit_score'] < 300) | (df['credit_score'] > 900), 'credit_score'] = pd.NA

## 4. Fill missing values and fix data types

I use the median for numeric columns because it is less affected by very large values. For text columns, I use the most common value. The `3+` dependents category is stored as 3 so the dependents column can be an integer column.

In [ ]:
# Fill numeric missing values with the median
numeric_columns = ['applicant_age', 'monthly_income_inr', 'loan_amount_inr', 'credit_score']
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Fill text missing values with the most common value
text_fill_columns = ['dependents', 'employment_type', 'property_area']
for column in text_fill_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

# Convert dependents to an integer column
df['dependents'] = df['dependents'].replace('3+', '3').astype(int)

df.info()

## 5. Check the cleaned data

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df['loan_status'].value_counts()

## 6. Save the clean data

The original CSV is left as it is. This creates a separate cleaned file in the same folder as this notebook.

In [ ]:
df.to_csv('../data/private/banking_loan_applications_cleaned.csv', index=False)